In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Claims section

In [0]:
claims = spark.table(
    "healthcare_dev.bronze.claims"
)

In [0]:
window_spec = (
    Window
    .partitionBy("claim_id")
    .orderBy(
        col("event_time").desc()
    )
)

claims_clean = (

    claims

    .withColumn(
        "rn",
        row_number().over(window_spec)
    )

    .filter(
        col("rn")==1
    )

    .drop("rn")
)

In [0]:
claims_late = (

    claims_clean

    .filter(
        col("is_late_simulated")==True
    )

)

claims_valid=(

    claims_clean

    .filter(
        col("is_late_simulated")==False
    )

    .drop(
        "is_late_simulated"
    )

)

In [0]:
claims_valid=(

claims_valid

.withColumn(
    "updated_at",
    col("ingestion_timestamp")
)

)

In [0]:
(
claims_valid.write

.format("delta")

.mode("overwrite")

.option(
    "overwriteSchema",
    "true"
)

.saveAsTable(
"healthcare_dev.silver.claims"
)
)

(
claims_late.write

.format("delta")

.mode("overwrite")

.saveAsTable(
"healthcare_dev.silver.claims_quarantine"
)
)

In [0]:
incoming_claims=(

spark.table(
"healthcare_dev.bronze.claims"
)

.withColumn(
"updated_at",
current_timestamp()
)

)

In [0]:
incoming_claims=(

incoming_claims

.withColumn(

"claim_amount",

when(

col("claim_id").isin(

"00001843-fe30-4a9f-9b9e-e3002013554f",
"0001abba-75b2-4ee7-9087-fd1db77da3f9",
"00021f77-1dfe-44f4-a161-bbfe8f0d9d13"

),

col("claim_amount")+200

)

.otherwise(
col("claim_amount")
)

)

)

In [0]:
window_spec=(

Window

.partitionBy(
"claim_id"
)

.orderBy(
col("updated_at").desc()
)

)

incoming_claims_dedup=(

incoming_claims

.withColumn(
"rn",
row_number().over(
window_spec
)
)

.filter(
col("rn")==1
)

.drop("rn")

)

In [0]:
silver=DeltaTable.forName(
spark,
"healthcare_dev.silver.claims"
)

(
silver.alias("target")

.merge(

incoming_claims_dedup.alias("source"),

"""
target.claim_id =
source.claim_id
"""

)

.whenMatchedUpdate(

condition=
"""
source.updated_at >
target.updated_at
""",

set={

"claim_amount":
"source.claim_amount",

"updated_at":
"source.updated_at"

}

)

.whenNotMatchedInsertAll()

.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
SELECT
    claim_id,
    COUNT(*) cnt
FROM healthcare_dev.silver.claims
    GROUP BY claim_id
    HAVING COUNT(*)>1

claim_id,cnt


In [0]:
%sql
select count(*) from healthcare_dev.silver.claims

count(*)
60970


In [0]:
%sql
describe healthcare_dev.silver.claims

col_name,data_type,comment
claim_id,string,null
patient_id,string,null
provider_id,string,null
claim_amount,double,null
currency,string,null
diagnosis_code,string,null
claim_status,string,null
claim_type,string,null
use,string,null
event_time,timestamp,null


### Patients section

In [0]:
patients = spark.table("healthcare_dev.bronze.patients")

In [0]:
display(patients.dtypes)

_1,_2
patient_id,string
gender,string
birth_date,string
city,string
state,string
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
patients = patients.withColumn("birth_date",col("birth_date").cast("date"))

In [0]:
display(patients.dtypes)

_1,_2
patient_id,string
gender,string
birth_date,date
city,string
state,string
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
patients.show(3)

+--------------------+------+----------+---------+-------------+--------------------+--------------------+---------+-------------+
|          patient_id|gender|birth_date|     city|        state|         source_file| ingestion_timestamp| batch_id|source_system|
+--------------------+------+----------+---------+-------------+--------------------+--------------------+---------+-------------+
|5cbc121b-cd71-442...|  male|1945-12-10|  Taunton|Massachusetts|Aaron697_Brekke49...|2026-05-23 07:17:...|batch_001|         FHIR|
|adccf2c3-9dc4-406...|  male|1946-03-29| Westford|Massachusetts|Aaron697_Stiedema...|2026-05-23 07:17:...|batch_001|         FHIR|
|31191928-6acb-4d7...|female|2002-10-24|Wakefield|Massachusetts|Abby752_Kuvalis36...|2026-05-23 07:17:...|batch_001|         FHIR|
+--------------------+------+----------+---------+-------------+--------------------+--------------------+---------+-------------+
only showing top 3 rows


In [0]:
window_spec_p = Window.partitionBy("patient_id").orderBy(col("ingestion_timestamp").desc())

In [0]:
patient_clean = (
    patients
    .withColumn("rn",row_number().over(window_spec_p))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
(
    patient_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("healthcare_dev.silver.patients")
)

In [0]:
%sql
desc healthcare_dev.silver.patients

col_name,data_type,comment
patient_id,string,null
gender,string,null
birth_date,date,null
city,string,null
state,string,null
source_file,string,null
ingestion_timestamp,timestamp,null
batch_id,string,null
source_system,string,null


### Providers Section

In [0]:
providers = spark.table("healthcare_dev.bronze.providers")

In [0]:
display(providers.dtypes)

_1,_2
provider_id,string
provider_name,string
city,string
state,string
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
win_spec_pr = Window.partitionBy("provider_id").orderBy(col("ingestion_timestamp").desc())

In [0]:
providers_clean = providers.withColumn("rn",row_number().over(win_spec_pr)).filter(col("rn") == 1).drop("rn")

In [0]:
providers_clean.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("healthcare_dev.silver.providers")

In [0]:
%sql
desc healthcare_dev.silver.providers

col_name,data_type,comment
provider_id,string,null
provider_name,string,null
city,string,null
state,string,null
source_file,string,null
ingestion_timestamp,timestamp,null
batch_id,string,null
source_system,string,null


### Observation Section

In [0]:
observations = spark.table("healthcare_dev.bronze.observations")

In [0]:
display(observations.dtypes)

_1,_2
observation_id,string
patient_id,string
observation_code,string
value,double
unit,string
effective_date,string
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
observations = observations.withColumn("effective_date",col("effective_date").cast("timestamp"))

In [0]:
display(observations.dtypes)

_1,_2
observation_id,string
patient_id,string
observation_code,string
value,double
unit,string
effective_date,timestamp
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
 win_spec_o = Window.partitionBy("observation_id").orderBy(col("effective_date").desc())

In [0]:
observations_clean = observations.withColumn("rn",row_number().over(win_spec_o)).filter(col("rn") == 1).drop("rn")

In [0]:
display(observations_clean.dtypes)

_1,_2
observation_id,string
patient_id,string
observation_code,string
value,double
unit,string
effective_date,timestamp
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
observations_clean.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("healthcare_dev.silver.observations")

### Encounters section

In [0]:
encounters = spark.table("healthcare_dev.bronze.encounters")

In [0]:
display(encounters.dtypes)

_1,_2
encounter_id,string
patient_id,string
status,string
class,string
start,string
end,string
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
encounters = encounters.withColumn("start",col("start").cast("timestamp")).withColumn("end",col("end").cast("timestamp"))

In [0]:
display(encounters.dtypes)

_1,_2
encounter_id,string
patient_id,string
status,string
class,string
start,timestamp
end,timestamp
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
win_spec_e = Window.partitionBy("encounter_id").orderBy(col("start").desc())

In [0]:
encounters_clean = encounters.withColumn("rn",row_number().over(win_spec_e)).filter(col("rn") == 1).drop("rn")
encounters_clean.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("healthcare_dev.silver.encounters")

In [0]:
%sql
desc healthcare_dev.silver.encounters

col_name,data_type,comment
encounter_id,string,null
patient_id,string,null
status,string,null
class,string,null
start,timestamp,null
end,timestamp,null
source_file,string,null
ingestion_timestamp,timestamp,null
batch_id,string,null
source_system,string,null


### Conditions section

In [0]:
conditions = spark.table("healthcare_dev.bronze.conditions")

In [0]:
conditions = conditions.withColumn("onset",col("onset").cast("timestamp"))

In [0]:
display(conditions.dtypes)

_1,_2
condition_id,string
patient_id,string
condition_code,string
clinical_status,string
onset,timestamp
source_file,string
ingestion_timestamp,timestamp
batch_id,string
source_system,string


In [0]:
win_spec_c = Window.partitionBy("condition_id").orderBy(col("ingestion_timestamp").desc())

In [0]:
conditions_clean = conditions.withColumn("rn",row_number().over(win_spec_c)).filter(col("rn") == 1).drop("rn")

In [0]:
conditions_clean.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("healthcare_dev.silver.conditions")

In [0]:
%sql
desc healthcare_dev.silver.conditions  

col_name,data_type,comment
condition_id,string,null
patient_id,string,null
condition_code,string,null
clinical_status,string,null
onset,timestamp,null
source_file,string,null
ingestion_timestamp,timestamp,null
batch_id,string,null
source_system,string,null
